# PowerCo – Exploratory Data Analysis

## Task 2: Exploratory Data Analysis and Data Cleaning

**Objective:** Understand the structure, data types, distributions and basic data quality of the PowerCo customer and historical pricing datasets, with particular attention to the churn indicator.

The analysis follows the starter notebook provided by Estelle and focuses on descriptive statistics and simple visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

%matplotlib inline
sns.set(color_codes=True)

---

## 1. Loading the data

The client supplied customer-level data and historical pricing data. The data description identifies `id` as the client company identifier and `churn` as the indicator of whether a client churned over the next three months.

In [ ]:
client_df = pd.read_csv('./client_data.csv')
price_df = pd.read_csv('./price_data.csv')

print("Client data shape:", client_df.shape)
print("Price data shape:", price_df.shape)

In [ ]:
client_df.head(3)

In [ ]:
price_df.head(3)

---

## 2. Descriptive statistics

### Data types

Reviewing data types helps identify numerical, categorical and date fields before further analysis. Date columns are initially loaded as objects and can be converted to datetime after inspection.

In [ ]:
client_df.info()

In [ ]:
price_df.info()

### Data type summary

The customer dataset contains numerical measures such as consumption, margins, forecast values and subscribed power, categorical/hash-coded fields such as sales channel and gas status, and several date fields. The pricing dataset contains a client ID, reference date and historical variable/fixed price fields.

The data description notes that some text fields are hashed; their commercial meaning is retained and they may therefore still have predictive value.

In [ ]:
# Convert date fields to datetime for analysis
client_date_cols = ['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']
price_date_cols = ['price_date']

for col in client_date_cols:
    client_df[col] = pd.to_datetime(client_df[col], errors='coerce')

for col in price_date_cols:
    price_df[col] = pd.to_datetime(price_df[col], errors='coerce')

print("Client date columns:")
print(client_df[client_date_cols].dtypes)
print("\nPrice date column:")
print(price_df[price_date_cols].dtypes)

### Data quality checks

In [ ]:
print("Missing values in client data:")
display(client_df.isnull().sum().sort_values(ascending=False))

print("\nMissing values in price data:")
display(price_df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows in client data:", client_df.duplicated().sum())
print("Duplicate rows in price data:", price_df.duplicated().sum())

The supplied datasets contain no pandas-null values and no duplicated rows. The data description also indicates that some fields contain the literal value `MISSING`; this is distinct from a pandas null and should be treated as a category rather than silently discarded at this stage.

---

## 3. Descriptive statistics

`describe()` provides counts, central tendency and spread for numerical variables and helps identify variables with large ranges or potential skew/outliers.

In [ ]:
client_df.describe().T

In [ ]:
price_df.describe().T

### Categorical variables

In [ ]:
categorical_cols = client_df.select_dtypes(include='object').columns

categorical_summary = pd.DataFrame({
    'data_type': client_df[categorical_cols].dtypes.astype(str),
    'unique_values': client_df[categorical_cols].nunique(dropna=False),
    'missing_values': client_df[categorical_cols].isna().sum()
})

categorical_summary

The customer dataset contains 14,606 client records. The `id` field is unique per client. `channel_sales`, `has_gas` and `origin_up` are categorical/hash-coded variables. The `churn` field is binary.

The churn distribution is imbalanced: approximately **90.3%** of clients did not churn and **9.7%** churned in the supplied dataset. This imbalance should be kept in mind in later modeling/evaluation.

In [ ]:
churn_counts = client_df['churn'].value_counts().sort_index()
churn_percentage = client_df['churn'].value_counts(normalize=True).sort_index() * 100

pd.DataFrame({
    'count': churn_counts,
    'percentage': churn_percentage.round(2)
}, index=['Retention (0)', 'Churn (1)'])

---

## 4. Data visualization

The visualizations below focus on simple distributions and the churn split, as recommended in the task instructions.

In [ ]:
def plot_stacked_bars(dataframe, title_, size_=(18, 10), rot_=0, legend_="upper right"):
    ax = dataframe.plot(
        kind="bar",
        stacked=True,
        figsize=size_,
        rot=rot_,
        title=title_
    )
    annotate_stacked_bars(ax, textsize=14)
    plt.legend(["Retention", "Churn"], loc=legend_)
    plt.ylabel("Company base (%)")
    plt.show()

def annotate_stacked_bars(ax, pad=0.99, colour="white", textsize=13):
    for p in ax.patches:
        value = str(round(p.get_height(), 1))
        if value == '0.0':
            continue
        ax.annotate(
            value,
            ((p.get_x() + p.get_width()/2) * pad - 0.05,
             (p.get_y() + p.get_height()/2) * pad),
            color=colour,
            size=textsize
        )

def plot_distribution(dataframe, column, ax, bins_=50):
    temp = pd.DataFrame({
        "Retention": dataframe[dataframe["churn"] == 0][column],
        "Churn": dataframe[dataframe["churn"] == 1][column]
    })
    temp[["Retention", "Churn"]].plot(kind='hist', bins=bins_, ax=ax, stacked=True)
    ax.set_title(f"Distribution of {column} by churn status")
    ax.set_xlabel(column)

In [ ]:
# Churn distribution
churn = client_df[['id', 'churn']].copy()
churn.columns = ['Companies', 'churn']

churn_total = churn.groupby('churn').count()
churn_percentage = churn_total / churn_total.sum() * 100

plot_stacked_bars(
    churn_percentage.transpose(),
    "Churning status",
    (5, 5),
    legend_="lower right"
)

In [ ]:
# Distributions of selected consumption variables
distribution_cols = [
    'cons_12m',
    'cons_gas_12m',
    'cons_last_month',
    'imp_cons'
]

fig, axs = plt.subplots(nrows=1, ncols=len(distribution_cols), figsize=(20, 5))

for ax, col in zip(axs, distribution_cols):
    plot_distribution(client_df, col, ax)

plt.tight_layout()
plt.show()

In [ ]:
# Distributions of key forecast price variables
price_cols = [
    'forecast_price_energy_off_peak',
    'forecast_price_energy_peak',
    'forecast_price_pow_off_peak'
]

fig, axs = plt.subplots(nrows=1, ncols=len(price_cols), figsize=(18, 5))

for ax, col in zip(axs, price_cols):
    plot_distribution(client_df, col, ax)

plt.tight_layout()
plt.show()

In [ ]:
# Historical price distributions
historical_price_cols = [
    'price_off_peak_var',
    'price_peak_var',
    'price_mid_peak_var',
    'price_off_peak_fix',
    'price_peak_fix',
    'price_mid_peak_fix'
]

fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))

for ax, col in zip(axs.flatten(), historical_price_cols):
    price_df[col].plot(kind='hist', bins=50, ax=ax)
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col)

plt.tight_layout()
plt.show()

---

## 5. Initial EDA observations

Based on the supplied datasets:

- The customer dataset contains **14,606 client records and 26 columns**.
- The historical pricing dataset contains **193,002 records and 8 columns**.
- The customer `id` is unique in the customer-level dataset.
- There are no duplicated rows in either dataset.
- There are no pandas-null values in the supplied data.
- Several categorical fields contain the literal value `MISSING`, which should be retained as an explicit category unless a later modeling decision requires another treatment.
- The customer consumption and margin variables have wide ranges and visibly skewed distributions, indicating that outliers and skewness may need attention in later analysis.
- Churn is an imbalanced binary outcome, with approximately **9.7% churn** and **90.3% retention**.
- The price variables also show different distributions across off-peak, peak and fixed-price measures.

These observations describe the data but do **not** by themselves establish that price sensitivity causes churn. Further feature engineering, statistical analysis and modeling are required to investigate that hypothesis.

## Conclusion

The EDA provides an initial understanding of PowerCo's customer and pricing data, including their data types, distributions and basic quality characteristics. The next stage can build on this understanding by engineering relevant pricing/customer features and testing the relationship between price changes and churn.